## Dagster Conversion notebook ##
- Convert all dbt project code
- develop automated scripts for generating SDAs

In [2]:
# imports
import os
import yaml
import re
import json
from subprocess import Popen, PIPE

In [3]:
# helper functions

def get_nodes(location):
    nodes= ""
    for dir in os.listdir(location):
        nodes += ' ' + dir
    return nodes

def get_files(location,folder_list):
    yml_files = []
    sql_files = []
    for dir in os.listdir(location):
        if dir in folder_list:
            dir_path = os.path.join(location, dir)
            for filename in os.listdir(dir_path):
                s=os.path.join(dir_path,filename)
                if filename.upper().endswith('.YML'):
                    yml_files.append(s)
                elif filename.upper().endswith('.SQL'):
                    sql_files.append(s)
    return yml_files, sql_files

def process_yml(yml_file,source_prefix,model_prefix):
    with open(yml_file,'r',encoding='utf-8') as yf:
        yml_content = yaml.safe_load(yf)
    try:
        for sources in yml_content["sources"]:
            try:
                for x in sources["tables"]:
                    # print(x)
                    if x["name"].find(source_prefix)==-1:
                        x["name"]= source_prefix + x["name"]
            except:
                print(f'No tables in source: {sources}, for file: {yml_file}.')
    except Exception as e: 
        print(f'No sources section in yaml: {yml_file}, error: {e}')
    
    try:
        for model in yml_content["models"]:
            if model["name"].find(model_prefix)==-1:
                model["name"]= model_prefix + model["name"]
    except Exception as e:
        print(f'No models for file: {yml_file}. Error: {e}')

    with open(yml_file,'w') as yfw:
            yaml.dump(yml_content,yfw,sort_keys=False,default_style=False,allow_unicode=True)

def process_sql(sql_file,source_prefix):
    search_str1 = '{{'
    search_str2 = 'source('
    search_str3 = ','
    search_str4 = '}}'
    new_file = ''
    skipped_files = []
    try:
        with open(sql_file,'r',encoding='utf-8') as f:
            try:
                for line in f:
                    # cleanup bad code - insert cleanup steps here
                    line = line.replace('{{ source ("','{{ source("')

                    # add prefix
                    chk1 = line.find(search_str1)
                    chk2 = line.find(search_str2,chk1)
                    chk3 = line.find(search_str3,chk2)
                    chk4 = line.find(search_str4,chk3)
                    if -1 not in [chk1,chk2,chk3,chk4] and chk1 < chk2 < chk3 < chk4:
                        str_parse = line[chk3+1:chk4]
                        print(str_parse)
                        table_name = str_parse.strip().strip(')').strip('"').strip("'")
                        if table_name.find(source_prefix)==-1:
                            new_table_name = source_prefix + table_name
                            line = line[:chk3] + line[chk3:chk4].replace(table_name,new_table_name) + line[chk4:]
                            print(table_name,new_table_name,line)
                    new_file = new_file + line
            except Exception as e:
                print(f'Line failed: {line}, error: {e}' )
                raise e

        with open(sql_file,'w',encoding='utf-8') as f:
            f.write(new_file)

    except Exception as e:
        print(f'file open failed for {sql_file}: {e}')
        raise e
     
def remove_prefix_yml(yml_file,source_prefix):
    with open(yml_file,'r',encoding='utf-8') as yf:
        yml_content = yaml.safe_load(yf)
    try:
        for sources in yml_content["sources"]:
            try:
                for x in sources["tables"]:
                    # print(x)
                    if x["name"].find(source_prefix) != -1:
                        x["name"]= x["name"].replace(source_prefix,'')
            except:
                print(f'No tables in source: {sources}, for file: {yml_file}.')
    except Exception as e: 
        print(f'No sources section in yaml: {yml_file}, error: {e}')

    with open(yml_file,'w') as yfw:
            yaml.dump(yml_content,yfw,sort_keys=False,default_style=False,allow_unicode=True)

def remove_prefix_sql(sql_file,source_prefix):
    search_str1 = '{{'
    search_str2 = 'source('
    search_str3 = ','
    search_str4 = '}}'
    new_file = ''
    try:
        with open(sql_file,'r',encoding='utf-8') as f:
            try:
                for line in f:
                    # cleanup bad code - insert cleanup steps here
                    line = line.replace('{{ source ("','{{ source("')

                    # add prefix
                    chk1 = line.find(search_str1)
                    chk2 = line.find(search_str2,chk1)
                    chk3 = line.find(search_str3,chk2)
                    chk4 = line.find(search_str4,chk3)
                    if -1 not in [chk1,chk2,chk3,chk4] and chk1 < chk2 < chk3 < chk4:
                        str_parse = line[chk3+1:chk4]
                        print(str_parse)
                        table_name = str_parse.strip().strip(')').strip('"').strip("'")
                        if table_name.find(source_prefix) != -1:
                            new_table_name = table_name.replace(source_prefix,'')
                            line = line[:chk3] + line[chk3:chk4].replace(table_name,new_table_name) + line[chk4:]
                            print(table_name,new_table_name,line)
                    new_file = new_file + line
            except Exception as e:
                print(f'Line failed: {line}, error: {e}' )
                raise e

        with open(sql_file,'w',encoding='utf-8') as f:
            f.write(new_file)

    except Exception as e:
        print(f'file open failed for {sql_file}: {e}')
        raise e
    
def uppercase_yml(yml_file):
    messages = ''
    with open(yml_file,'r',encoding='utf-8') as yf:
        yml_content = yaml.safe_load(yf)
        # print(yml_content)
    messages += f'begin processing {yml_file}. \n'
    update_file = False
    messages += f'begin processing sources for {yml_file}. \n'
    try:
        for source in yml_content["sources"]:
            try:
                process_name = False
                for char in source["name"]:
                    if char.islower():
                        process_name = True
                if process_name:
                    messages += f'changing source: {source["name"]} \n'
                    source["name"]= source["name"].upper()
                    update_file = True
                    messages += f'new source value: {source["name"]} \n'
                else:
                    messages += f'no change for source: {source["name"]} \n'

                process_table_name = False
                for x in source["tables"]:
                    for char in x["name"]:
                        if char.islower():
                            process_table_name = True
                    if process_table_name:
                        messages += f'processing table: {x["name"]} \n'
                        x["name"]= x["name"].upper()
                        update_file = True
                        messages += f'new  table value: {x["name"]} \n'
                    else: 
                        messages += f'no changes to table: {source["name"]} \n'
            except:
                messages += f'No tables in source: {source}, for file: {yml_file}. \n'
    except Exception as e: 
        messages += f'No sources section in yaml: {yml_file}, error: {e} \n'
    
    messages += f'begin processing models for {yml_file}. \n'
    try:
        for model in yml_content["models"]:
            process_name = False
            try:
                for char in model["name"]:
                    if char.islower():
                        process_name = True
                if process_name:
                    messages += f'processing: {model["name"]} \n'
                    model["name"]= model["name"].upper()
                    update_file = True
                    messages += f'new value: {model["name"]} \n'
                else:
                    messages += f'no change to model: {model["name"]} \n'
            except:
                messages += f'No models in file: {yml_file}. \n'
    except Exception as e: 
        messages += f'No models section in yaml: {yml_file}, error: {e} \n'

    if update_file:
        messages += f'updated file: {yml_file}. \n'
        with open(yml_file,'w') as yfw:
                yaml.dump(yml_content,yfw,sort_keys=False,default_style=False,allow_unicode=True)
    return messages

def uppercase_sql(sql_file):
    search_jinj_open = '{{'
    search_src = 'source('
    search_ref = 'ref('
    search_comma = ','
    search_paren = ')'
    search_jinj_close = '}}'
    new_file = ''
    update_file = False
    messages = ''
    try:
        messages += f'begin processing lines for: {sql_file}. \n'
        with open(sql_file,'r',encoding='utf-8') as f:
            try:
                for line in f:
                    # cleanup bad code - insert cleanup steps here
                    original_line = line
                    line_updated = False
                    line = line.replace('{{ source ("','{{ source("')

                    # search for sources
                    src_chk1 = line.find(search_jinj_open)
                    src_chk2 = line.find(search_src,src_chk1)
                    src_chk3 = line.find(search_comma,src_chk2)
                    src_chk4 = line.find(search_paren,src_chk3)
                    src_chk5 = line.find(search_jinj_close,src_chk4)
                    
                    # search for refs
                    ref_chk1 = line.find(search_jinj_open)
                    ref_chk2 = line.find(search_ref,ref_chk1)
                    ref_chk3 = line.find(search_paren,ref_chk2)
                    ref_chk4 = line.find(search_jinj_close,ref_chk3)

                    # modify source matches
                    if -1 not in [src_chk1,src_chk2,src_chk3,src_chk4,src_chk5] and src_chk1 < src_chk2 < src_chk3 < src_chk4 < src_chk5:
                        messages += f'found source in line: {line} \n'
                        line_updated = True
                        str_parse_table = line[src_chk3+1:src_chk4]
                        str_parse_source = line[src_chk2+7:src_chk3]
                        # print(line,str_parse_source,str_parse_table,src_chk2,src_chk3)
                        source_name = str_parse_source.strip().strip("'").strip('"')
                        new_source_name = source_name.upper()
                        table_name = str_parse_table.strip().strip(')').strip('"').strip("'")
                        new_table_name = table_name.upper()
                        line = line[:src_chk3].replace(source_name,new_source_name) + line[src_chk3:src_chk4].replace(table_name,new_table_name) + line[src_chk4:]
                        messages += f'old table: {table_name}, new table: {new_table_name}, new line: {line}'
                    # modify ref matches
                    if -1 not in [ref_chk1,ref_chk2,ref_chk3,ref_chk4] and ref_chk1 < ref_chk2 < ref_chk3 < ref_chk4:
                        str_parse_ref = line[ref_chk2+4:ref_chk3]
                        # print(line,str_parse_ref,ref_chk2,ref_chk3)
                        messages += f'found ref in line: {line}, {str_parse_ref} \n'
                        line_updated = True
                        ref_name = str_parse_ref.strip().strip("'").strip('"')
                        new_ref_name = ref_name.upper()
                        line = line[:ref_chk3].replace(ref_name,new_ref_name) +  line[ref_chk3:]
                        messages += f'old ref: {ref_name}, new ref: {new_ref_name}, new line: {line}'
                    new_file = new_file + line
                    if line != original_line: update_file = True
            except Exception as e:
                messages += f'Line failed: {line}, error: {e} \n' 
                raise e
        if update_file:
            messages += f'updated lines for: {sql_file}. \n'
            with open(sql_file,'w',encoding='utf-8') as f:
                f.write(new_file)

    except Exception as e:
        messages += f'file open failed for {sql_file}: {e} \n'
        raise e
    
    return messages

def powershell_execute(script_input):
    CreatedCommandArray = ["powershell.exe"] + script_input
    # print(CreatedCommandArray)
    p = Popen(CreatedCommandArray, stdout=PIPE, stderr=PIPE, shell=True)
    output, error = p.communicate()
    return(output,error, CreatedCommandArray)

In [ ]:
# remove source prefix from tables in models files
project_location=rf"C:\Source\Repos\EnterpriseCloud\dbt\ods_to_pub\models"
folder_list= ["telcustomer"]
folder_list1 = ["activedirectory","ariba","baan","baan_tee","carma","crm","crmhelpdesk","crmsales","crmservice"]
folder_list2 = ["crm_file","csdpro","file","gem","kronos","lansweeper","legacy93","legacyprod1","mdm","monday"]
folder_list3 = ["msdynamics","myskills","now","nowuat","prefect","prophet","redgate","rosetta","sapapi","saph","sapo"]
folder_list4 = ["sharepoint","sirfis","snowflake","teacommon","telcustomer","tgis","threepl","validator","windchill","wisdom","workday"]
yml_files, sql_files = get_files(project_location,folder_list)
# print(yml_files)
# print(sql_files)
source_prefix = 'ODS_'

# process yml files
for yml_file in yml_files:
    print(yml_file)
    remove_prefix_yml(yml_file,source_prefix)
    # break

# process sql files
for sql_file in sql_files:
    print(sql_file)
    remove_prefix_sql(sql_file,source_prefix)
    # break


In [ ]:
# update LZ source names
project_location=rf"C:\Source\Repos\EnterpriseCloud\dbt\ods_to_common\models"
source_prefix = 'ODS_'
model_prefix = 'COMMON_'
yml_files, sql_files = get_files(project_location)
print(sql_files)

# process yml files
for yml_file in yml_files:
    print(yml_file)
    process_yml(yml_file,source_prefix,model_prefix)
    # break

# proces sql files
for sql_file in sql_files:
    print(sql_file)
    process_sql(sql_file,source_prefix)
    # break
  

In [26]:
#get list of folders/nodes for the dbt asset ingestion
project_location=rf"C:\Source\Repos\EnterpriseCloud\dbt\pub_to_pub\models"
nodes = get_nodes(project_location)
print(nodes.replace(' ','","'))

","de_field_service","de_fst_departmental","de_intel_sales_solutions","de_manufacturing","de_sales_operations","de_tea_support_dm","de_tel_reporting","reports","schema.yml


In [66]:
# get out list and return string for source asset.py files
group = 'lz_telcustomer'
schema_asset_key = 'TELCUSTOMER'
ConfigFileName = 'C:\\Source\Repos\\EnterpriseCloud\\Configuration\\TELCustomer.json'
table_list = []
return_string = "return('"
out_string = "outs = {'"
out_tail = f"': AssetOut(key_prefix=[CODE_SOURCE,'LZ','{schema_asset_key}'],group_name='{group}'),\n'"
with open(ConfigFileName) as config_file:
    data = json.load(config_file)
config = None
for config, vals in data["FileUploader"].items():
    # print(config)
    table_name = vals["SNOWFLAKE_TABLE_NAME"]
    print(table_name)
    table_list.append(table_name.upper())
table_set = set(table_list)

for tbl in table_set:
    return_string = return_string + tbl + "','"
    out_string = out_string + tbl + out_tail

print(out_string)
print(return_string)

CDI_VISIT
CDI_PAGEVIEW
USER_INFO
BUSINESS_PARTNER
UGK
PRODUCT
CONTACT
CAPITAL_PO_CHANGE_LOG
CAPITAL_PO_CHANGE_TYPE
CAPITAL_PO_COMMENTS
CAPITAL_PO_EVENT_TRACKING
CAPITAL_PO_HEADER
CAPITAL_PO_LINE
CAPITAL_PO_QUOTE_MAP
CAPITAL_PO_REASON_CODE
CAPITAL_PO_SHIP_TO_DESCR
CAPITAL_PO_SHIP_TO_LOCATION
CAPITAL_PO_VALIDATION_LOG
SHIPPING_ADVICE
TELCUST_ARF_NEW_QUOTE
TELCUST_ARF_QRF_DATA
CAPITAL_PO_HEADER_MAX_DATE_test
EQUIPMENT_QUOTE_TRACKING
outs = {'CAPITAL_PO_CHANGE_LOG': AssetOut(key_prefix=[CODE_SOURCE,'LZ','TELCUSTOMER'],group_name='lz_telcustomer'),
'CAPITAL_PO_SHIP_TO_LOCATION': AssetOut(key_prefix=[CODE_SOURCE,'LZ','TELCUSTOMER'],group_name='lz_telcustomer'),
'CAPITAL_PO_HEADER': AssetOut(key_prefix=[CODE_SOURCE,'LZ','TELCUSTOMER'],group_name='lz_telcustomer'),
'CAPITAL_PO_SHIP_TO_DESCR': AssetOut(key_prefix=[CODE_SOURCE,'LZ','TELCUSTOMER'],group_name='lz_telcustomer'),
'CAPITAL_PO_QUOTE_MAP': AssetOut(key_prefix=[CODE_SOURCE,'LZ','TELCUSTOMER'],group_name='lz_telcustomer'),
'TELCUST_ARF_N

In [ ]:
# TelTools extract tester - use for testing prior to local import
# change environment intepreter to dagster(python 3.11.4)
import os
from TELTools import FileUploader, BuildCommand, TTPowershellExtract


############ build source ingestion assets
UploaderConfigFileName = 'F:\\Shares\\Configuration\\FileUploader_telcustomer.json'
CODE_SOURCE = os.environ['COMPUTERNAME']
DataSetName = ["UserList","GroupList","GroupMemberList"]
TTConfigFileName = "TELTools.json"
SourceName = "AzActiveDirectory"

for DataSet in DataSetName:
    print(DataSet)
    TELToolsScript=BuildCommand(SourceName=SourceName,DataSetName=DataSet,ConfigFileName=TTConfigFileName)
    print(TELToolsScript)
    extract_message = TTPowershellExtract(TELToolsScript=TELToolsScript)
    print(extract_message)
    break

# # add file uploader code
# uploader_message = FileUploader(ConfigFileName=UploaderConfigFileName)
# print(extract_message,uploader_message) #TODO - pass messages back to dagster logs
   

In [ ]:
# convert database objects to upper case by source
# moves source name and table names to upper case in schema.yml and model.sql files
dbt_location="C:\\Source\\Repos\\EnterpriseCloud\\dbt\\"
project = "pub_to_pub" # change to switch projects
project_path = dbt_location + project + "\\models"
source_folder_lz = ["ariba","baan","baan_tee","carma","crm","crmhelpdesk","crmsales","crmservice","crm_file","csdpro","file","gem","lansweeper","legacy93","legacyprod1","mdm","monday","msdynamics","myskills","now","nowuat","prefect","prophet","redgate","rosetta","sapapi","saph","sapo","sharepoint","sirfis","snowflake","teacommon","tgis","threepl","Validator","windchill","wisdom","workday"]
# source_folder_lz = ["saph"]
source_folder_ods = ["activedirectory","ariba","baan","baan_tee","carma","crm","crmhelpdesk","crmservice","csdpro","de_field_service","de_fst_departmental","de_helpdesk","de_intel_sales_solutions","de_manufacturing","de_sales_operations","de_saph","de_spares_revenue","de_tea_support_dm","de_tee_sales_service_admin","de_tel_reporting","de_threepl","de_wisdom","gem","kronos","legacy93","mdm","monday","myskills","now","prefect","prophet","reports","rosetta","saph","sapo","shared_datasets","sharepoint","sirfis","teacommon","telcustomer","tgis","threepl","Validator","windchill","wisdom","workday"]
source_folder_odspub = ["mdm","monday","msdynamics","myskills","now","prefect","prophet","redgate","sapapi","saph","sapo","sharepoint","sirfis","teacommon","telcustomer","threepl","workday"]
source_folder_commonpub = ["activedirectory","baan","baan_tee","carma","common_system","crm","crmhelpdesk","crmsales","crmservice","csdpro","de_business_operations","de_business_partner_master","de_field_service","de_finance","de_fst_departmental","de_global_information_support","de_helpdesk","de_indirect_procurement","de_intel_sales_solutions","de_intel_sms_specialist","de_manufacturing","de_operations","de_planning","de_procurement_department","de_sales","de_sales_operations","de_saph","de_sap_master_data","de_sap_revenue_cogs","de_service","de_spares_revenue","de_tea_support_dm","de_tee_sales_service_admin","de_tel_reporting","de_threepl","de_warehouse_operations","de_windchill","kronos","legacy93","mdm","monday","myskills","now","prefect","prophet","rosetta","saph","sapo","sharepoint","sirfis","teacommon","telcustomer","tgis","threepl","Validator","wisdom","workday"]
source_folder_pub = ["de_field_service","de_fst_departmental","de_intel_sales_solutions","de_manufacturing","de_sales_operations","de_tea_support_dm","de_tel_reporting","reports"]

yml_files, sql_files = get_files(project_path,source_folder_pub)

# print(yml_files)
# print(sql_files)
out_file = f'..\\case_conv_{project}.txt'
file_out = ''
for yml_file in yml_files:
    print(f'processing: {yml_file}')
    file_out += uppercase_yml(yml_file=yml_file)

for sql_file in sql_files:
    print(f'processing: {sql_file}')
    file_out += uppercase_sql(sql_file=sql_file)
    file_name = os.path.basename(sql_file)
    path_name = os.path.dirname(sql_file)
    sql_file_upper = os.path.join(path_name,file_name.upper())
    if sql_file != sql_file_upper:
        script_input = []
        script_input += [f'git mv {sql_file} {sql_file_upper}; \n']
        file_out += str(script_input)
        output, error, command_array = powershell_execute(script_input=script_input)
        file_out += f'powershell output: {output},{error},{command_array} \n'

with open(out_file,'w') as out:
    out.write(file_out) 

In [ ]:
# pub_to_pub schema.yml file
yml_file = 'C:\\Source\\Repos\\EnterpriseCloud\\dbt\\pub_to_pub\\models\\schema.yml'
print(uppercase_yml(yml_file=yml_file))

In [1]:
import sys
print(sys.path)

['c:\\Source\\Repos\\EnterpriseCloud\\Notebooks', 'C:\\Source\\Repos\\EnterpriseCloud\\Python', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37\\python37.zip', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37\\DLLs', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37\\lib', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37', '', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37\\lib\\site-packages', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37\\lib\\site-packages\\win32', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37\\lib\\site-packages\\win32\\lib', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37\\lib\\site-packages\\Pythonwin', 'c:\\Users\\214225\\AppData\\Local\\Programs\\Python\\Python37\\lib\\site-packages\\IPython\\extensions', 'C:\\Users\\214225\\.ipython']


In [ ]:
# testing teltool.py for new source.json config file
import os
from TELTools import FileUploader, BuildCommand, TTPowershellExtract, getFileUploaderConfig
ConfigFileName = "TELCustomer.json"
# ConfigFileName = "TELTools.json"
# with open(ConfigFileName) as config_file:
#         data = json.load(config_file)
# print(data)
# for d,config in data["FileUploader"].items():
#         print(d)
#         print(config)

SourceName = 'SQLStatements'
DataSetName = "TELCustomer_Reporting-CAPITAL_PO_LINE"


TELToolsScript=BuildCommand(SourceName=SourceName,DataSetName=DataSetName,ConfigFileName=ConfigFileName)
print(TELToolsScript)
extract_message = TTPowershellExtract(TELToolsScript=TELToolsScript)
print(extract_message)

# config = getFileUploaderConfig(ConfigFileName)
# config = config["SQLStatements"]["DataSets"]
# del config["FunctionName"]
# print(config)

# for DataSetName,DataSetValues in config.items():
#     print(DataSetValues["OutputFile"]["FileName"])



############ build source ingestion assets
# ConfigFileName = 'TelCustomer.json'
# ENTERPRISECLOUD_REPO = os.environ.get('ENTERPRISECLOUD_REPO')
# print(ENTERPRISECLOUD_REPO)
# ConfigFileNamePath = os.path.join(ENTERPRISECLOUD_REPO,"Configuration",ConfigFileName)
# print(ConfigFileNamePath)
# config = getFileUploaderConfig(ConfigFileNamePath)
# config = config["SQLStatements"]["DataSets"]
# print(config)